In [1]:
import numpy as np

## Ridge Regression: 
####       it has closed form solution,
####       The solution is obtained by minimizing 
###        RSS(weights,alpha)= MSE+ alpha*(sum of squared weights)

In [2]:
class Ridge:
    def __init__(self,alpha):
        self.weights=None
        self.alpha=alpha
    def predict(self,X):
        X=np.concatenate((np.ones((X.shape[0],1)),X),axis=1)
        prediction=X@self.weights
        return prediction
    def fit(self,X,y):
        X=np.concatenate((np.ones((X.shape[0],1)),X),axis=1)
        a=X.T@X+self.alpha*np.identity(X.shape[1])
        b=X.T@y
        self.weights=np.linalg.inv(a)@b
    def MSE(self,y,y_hat):
        mse=0
        for r,s in zip(y,y_hat):
            mse+=(r-s)**2
        return float(mse)
    def __repr__(self,):
        return f"Ridge Regressor :\n -> Alpha = {self.alpha}\n -> Weights = {self.weights} "

In [3]:
model=Ridge(3)
X=np.array(
    [[2,4,6],
    [3,6,9],
    [4,8,12]]
)
y=np.array([2,3,4])
model.fit(X,y)

In [4]:
model.predict(np.array([[5,10,15]]))

array([4.95227273])

In [5]:
lam=np.arange(0.25,10,0.25)
best_model=None
best_mse=float('inf')
for l in lam:
    model=Ridge(l)
    model.fit(X,y)
    y_hat=model.predict(X)
    mse=model.MSE(y,y_hat)
    print(model)
    if mse<best_mse:
        best_model=model
        best_mse=mse
print("*"*60)
print(f"Best model is \n {best_model}")
print("*"*60)

Ridge Regressor :
 -> Alpha = 0.25
 -> Weights = [0.01207648 0.07111707 0.14223415 0.21335122] 
Ridge Regressor :
 -> Alpha = 0.5
 -> Weights = [0.01558442 0.07099567 0.14199134 0.21298701] 
Ridge Regressor :
 -> Alpha = 0.75
 -> Weights = [0.01724964 0.07091519 0.14183038 0.21274557] 
Ridge Regressor :
 -> Alpha = 1.0
 -> Weights = [0.01821862 0.0708502  0.1417004  0.21255061] 
Ridge Regressor :
 -> Alpha = 1.25
 -> Weights = [0.01885014 0.07079275 0.14158551 0.21237826] 
Ridge Regressor :
 -> Alpha = 1.5
 -> Weights = [0.0192926  0.07073955 0.1414791  0.21221865] 
Ridge Regressor :
 -> Alpha = 1.75
 -> Weights = [0.01961853 0.07068898 0.14137797 0.21206695] 
Ridge Regressor :
 -> Alpha = 2.0
 -> Weights = [0.01986755 0.07064018 0.14128035 0.21192053] 
Ridge Regressor :
 -> Alpha = 2.25
 -> Weights = [0.02006316 0.07059261 0.14118521 0.21177782] 
Ridge Regressor :
 -> Alpha = 2.5
 -> Weights = [0.02022018 0.07054594 0.14109189 0.21163783] 
Ridge Regressor :
 -> Alpha = 2.75
 -> Weight

## LASSO Regression :
#### Lasso doesn't have any closed form solutio.
#### We continously reweight such that it minimizes the loss function.
#### For lasso the loss function is
###   RSS(weights,alpha)=MSE+alpha*(sum of absolute values of weights)
###   $$J(w) = \text{MSE} + \alpha \sum_{j=1}^{d} |w_j|$$
#### Lasso can converge a weight to zero, that means it can also identify the likely unnecessary variables

## Approach 1: Subgradient Descent

### Mathematical Overview
### Because $|w|$ has a sharp corner at zero, we use the subgradient of the penalty term, which translates directly to the element-wise sign of the weights:
## $$\nabla \Omega(w) = \alpha \cdot \text{sign}(w)$$

### Implementation

In [6]:
class LassoSubgradientDescent:
    def __init__(self,alpha):
        self.alpha=alpha
        self.weights=None
    def predict(self,X):
        X=np.concatenate((np.ones((X.shape[0],1)),X),axis=1)
        prediction=X@self.weights
        return prediction
    def MSE(self,y,y_hat):
        mse=0
        for r,s in zip(y,y_hat):
            mse+=(r-s)**2
        return mse
    def fit(self,X,y,max_iter,learning_rate):
        self.weights=np.random.rand(X.shape[1]+1)
        
        X_expanded=X=np.concatenate((np.ones((X.shape[0],1)),X),axis=1)
        for _ in range(max_iter):
            cost=self.MSE(y,y_hat)+self.alpha*np.sum(np.abs(self.weights))
            p=X_expanded@self.weights
            w_grad_mse=(-2*X_expanded.T@(y-p))/len(y)
            w_grad_penality=self.alpha*np.sign(self.weights)
            grad=w_grad_mse+w_grad_penality
            self.weights-=learning_rate*grad
            print(f"Iternation {_+1} | Loss = {cost}")
            
    def __repr__(self,):
        return f"Lasso Regressor :\n -> Alpha = {self.alpha}\n -> Weights = {self.weights} "

In [7]:
X_test = np.array([
    [1.0, 2.0, 3.0],
    [2.0, 4.0, 5.0],
    [3.0, 5.0, 1.0],
    [4.0, 1.0, 2.0]
])

# True relationship: y = 5 + 2*x1 + 0*x2 - 3*x3
y_test = np.array([0.0, 0.0, 8.0, 7.0]) 
model = LassoSubgradientDescent(alpha=0.1)
model.fit(X_test, y_test, max_iter=80000, learning_rate=0.00000001)

Iternation 1 | Loss = 29.34872834257264
Iternation 2 | Loss = 29.348728358740132
Iternation 3 | Loss = 29.348728374907612
Iternation 4 | Loss = 29.34872839107509
Iternation 5 | Loss = 29.348728407242554
Iternation 6 | Loss = 29.348728423410016
Iternation 7 | Loss = 29.348728439577467
Iternation 8 | Loss = 29.34872845574491
Iternation 9 | Loss = 29.34872847191235
Iternation 10 | Loss = 29.34872848807978
Iternation 11 | Loss = 29.3487285042472
Iternation 12 | Loss = 29.348728520414614
Iternation 13 | Loss = 29.348728536582023
Iternation 14 | Loss = 29.34872855274942
Iternation 15 | Loss = 29.348728568916815
Iternation 16 | Loss = 29.3487285850842
Iternation 17 | Loss = 29.348728601251576
Iternation 18 | Loss = 29.348728617418946
Iternation 19 | Loss = 29.34872863358631
Iternation 20 | Loss = 29.348728649753664
Iternation 21 | Loss = 29.348728665921012
Iternation 22 | Loss = 29.34872868208835
Iternation 23 | Loss = 29.348728698255684
Iternation 24 | Loss = 29.348728714423007
Iternation 25

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [8]:
print(model)

Lasso Regressor :
 -> Alpha = 0.1
 -> Weights = [0.20709741 0.36751391 0.20973374 0.34608018] 


In [9]:
lam=np.arange(0.25,10,0.25)
best_model=None
best_cost=float('inf')
for l in lam:
    model=LassoSubgradientDescent(l)
    model.fit(X,y,max_iter=200,learning_rate=0.001)
    print(model)
    y_hat=model.predict(X)
    cost=model.MSE(y,y_hat)+model.alpha*np.sum(np.abs(model.weights))
    if cost<best_cost:
        best_model=model
        best_cost=cost
print("*"*60)
print(f"Best model is \n {best_model}")
print(f"Cost value of best_model is {best_cost}")
print("*"*60)

Iternation 1 | Loss = 0.5308330831723862
Iternation 2 | Loss = 0.47129231037109853
Iternation 3 | Loss = 0.4279559100643904
Iternation 4 | Loss = 0.39640771187425766
Iternation 5 | Loss = 0.3939830215037339
Iternation 6 | Loss = 0.39291285574069856
Iternation 7 | Loss = 0.3920637511557691
Iternation 8 | Loss = 0.3913754646872372
Iternation 9 | Loss = 0.39080417129526956
Iternation 10 | Loss = 0.39031798956368446
Iternation 11 | Loss = 0.3898937267080376
Iternation 12 | Loss = 0.3895145106656079
Iternation 13 | Loss = 0.38916806751113786
Iternation 14 | Loss = 0.38884546832790967
Iternation 15 | Loss = 0.38854021759364227
Iternation 16 | Loss = 0.38824759000828873
Iternation 17 | Loss = 0.38796414805593904
Iternation 18 | Loss = 0.38768739104542427
Iternation 19 | Loss = 0.38741549979776996
Iternation 20 | Loss = 0.38714715091389823
Iternation 21 | Loss = 0.3868813816598908
Iternation 22 | Loss = 0.3866174916750297
Iternation 23 | Loss = 0.38635497146731307
Iternation 24 | Loss = 0.3860

## Approach 2: Coordinate Descent (Analytical Selection)

### Mathematical Overview
#### Instead of updating all weights simultaneously, Coordinate Descent optimizes **one weight at a time** while holding all others constant. It calculates the partial residual $\rho_j$ (the error left over excluding feature $j$) and applies the **Soft-Thresholding Operator**:

## $$S(\rho_j, \alpha) = \text{sign}(\rho_j) \max(|\rho_j| - \alpha, 0)$$

#### This forces uninformative feature weights to snap cleanly to exactly `0.0`.

### Implementation